# Chapter 3 Walkthrough: A Synthetic Lab for Real Pharma Questions

It is your first Monday on the Roventra brand analytics team, and the brand director wants to know why December claims are down by a third. You will confirm the number, almost send it, and then discover that *nothing went wrong in December*. The claims simply had not arrived yet.

This notebook replays the chapter as a single executable story. Run the cells in order with the default generated data (`seed=20260609`) and your outputs will match the book exactly.

**Learning contract:** by the end of this notebook you will have:
- stopped a false trend report with a maturity rule
- reconstructed PAT02034's treatment attempt across three patient-linked sources
- derived completed fills from raw pharmacy transactions
- selected the payer rule active on an event date
- run a quality gate and built A1C patient groups

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT   = Path(".").resolve().parent
DATA   = ROOT / "ch03_data" / "output_data" / "generated_data"
SCRIPTS = ROOT / "ch03_data" / "scripts"
sys.path.insert(0, str(SCRIPTS))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.2f}".format)
print("Setup complete. DATA =", DATA)

Setup complete. DATA = /Users/qiu/Projects/hands-on-pharma-decision-science/ch03_data/output_data/generated_data


## 1. Generate the data package

One command builds everything. Run it from the project root if you have not already:

```bash
uv run python ch03_data/scripts/generate_all_synthetic_data.py
```

If counts match those in the book, reproducibility is confirmed.

**Listing 3.1: Verify the generated package contract**

In [2]:
import json

with open(DATA / "manifest.json") as f:
    manifest = json.load(f)

print("schema_version :", manifest["schema_version"])
print("seed           :", manifest["run_config"]["seed"])
rx_range = manifest["table_contracts"]["claims_pharmacy/pharmacy_claims.csv"]["date_ranges"]["fill_date"]
print("pharmacy fills :", rx_range["min"], "to", rx_range["max"])

schema_version : 2026-06-11
seed           : 20260609
pharmacy fills : 2024-01-01 to 2025-01-31


**What to notice:** `schema_version` ties this package to a specific generator release. `seed` guarantees exact reproducibility. `pharmacy fills` is the as-of cutoff: every trend read must respect this date.

## 2. The false December decline

Before touring individual sources, reproduce the chapter opening trap. It is January 5, 2025.

**Listing 3.2: Measure claim completeness at an as-of date**

In [3]:
import pandas as pd

mc = pd.read_csv(
    DATA / "claims_medical" / "medical_claims.csv",
    parse_dates=["claim_date", "claim_received_date"],
)
mc = mc[~mc.is_duplicate & mc.claim_date.dt.year.eq(2024)]
month = mc.claim_date.dt.to_period("M").astype(str).rename("service_month")

snapshot = month[mc.claim_received_date.le("2025-01-05")].value_counts().sort_index()
mature   = month.value_counts().sort_index()
view = pd.DataFrame({"snapshot_jan05": snapshot, "eventually": mature})
view["completeness_%"] = (100 * view.snapshot_jan05 / view.eventually).round(1)
view.tail(4)

,snapshot_jan05,eventually,completeness_%
service_month,,,
2024-09,3697,3697,100.00
2024-10,3956,3999,98.90
2024-11,3670,3870,94.80
2024-12,2701,3990,67.70


**Conclusion:** at the January 5 snapshot, December reads 32.3% low. The `eventually` column shows the full picture: 3,990 claims arrive, in line with every other month. The correction is a maturity rule -- exclude immature months or mark them incomplete before sharing any trend.

## 3. Five sources, five questions

Meet **PAT02034**: a generated female patient, 50-64 years old, in New Jersey, managed by HCP0280 (Endocrinology, ACC089) under commercial payer PAY002.

In [4]:
patients = pd.read_csv(DATA / "reference" / "patients.csv")
patients.head(3)

,patient_id,account_id,hcp_id,payer_id,state,region,age_band,sex,condition_bucket,coverage_start,coverage_end
0,PAT00001,ACC250,HCP0496,PAY002,NJ,Northeast,50-64,F,Rheumatology,2023-01-13,2025-10-18
1,PAT00002,ACC042,HCP0162,PAY007,WA,West,50-64,F,Cardiology,2023-12-19,2025-09-04
2,PAT00003,ACC089,HCP0245,PAY001,NY,Northeast,65+,F,Launch condition,2023-02-28,2025-02-21


**Listing 3.3: Build PAT02034's event ledger**

In [5]:
import pandas as pd

ehr = pd.read_csv(DATA / "ehr" / "ehr_events.csv")
sp = pd.read_csv(DATA / "specialty_pharmacy" / "specialty_pharmacy_events.csv")
rx = pd.read_csv(DATA / "claims_pharmacy" / "pharmacy_claims.csv", dtype={"ndc": str})

ehr_rows = (ehr.loc[ehr.patient_id.eq("PAT02034")]
    .assign(source="EHR", event=lambda d: d["event_type"], status=lambda d: d["event_type"])
    [["event_date", "source", "event", "status"]])
sp_row = sp.loc[sp.patient_id.eq("PAT02034")].iloc[0]
sp_rows = pd.DataFrame([
    {"event_date": sp_row["referral_date"], "source": "SP", "event": "Referral", "status": "Open"},
    {"event_date": sp_row["auth_decision_date"], "source": "SP", "event": "Authorization", "status": sp_row["auth_status"]},
    {"event_date": sp_row["ship_date"], "source": "SP", "event": "Shipment", "status": sp_row["delivery_status"]},
])
rx_rows = (rx.loc[rx.patient_id.eq("PAT02034") & rx.ndc.eq("90000-1001-11")]
    .rename(columns={"fill_date": "event_date"})
    .assign(source="Pharmacy", event=lambda d: d["reject_reason"].fillna("").where(d.status.eq("Rejected"), "Fill transaction"))
    [["event_date", "source", "event", "status"]])
ledger = pd.concat([ehr_rows, sp_rows, rx_rows], ignore_index=True).sort_values("event_date").reset_index(drop=True)
print(ledger.to_string(index=False))

event_date   source                                                  event    status
2024-03-07      EHR                                              Encounter Encounter
2024-06-12      EHR                                              Encounter Encounter
2024-06-13      EHR                                                    Lab       Lab
2024-06-13       SP                                               Referral      Open
2024-06-18       SP                                          Authorization  Approved
2024-06-22      EHR                                                    Lab       Lab
2024-07-02 Pharmacy New-to-market review; exception documentation required  Rejected
2024-07-09 Pharmacy                                       Fill transaction      Paid
2024-07-10       SP                                               Shipment   Shipped
2024-08-09 Pharmacy                                       Fill transaction      Paid
2024-08-10 Pharmacy                                       Fill tr

**What to notice:** the July 2 pharmacy row names the new-to-market exception barrier. The July 9 paid resubmission and July 10 shipment complete the access path introduced in Chapter 2.

### 3.5.1 Pharmacy claims: how many fills did PAT02034 actually complete?

In [6]:
ndc = pd.read_csv(DATA / "reference" / "ndc_codes.csv", dtype={"NDC": str})
name_map = dict(zip(ndc["NDC"], ndc["CUI_L1_NAME"]))
pat = rx.loc[rx.patient_id.eq("PAT02034") & rx.ndc.eq("90000-1001-11")].copy()
pat["product_name"] = pat["product_name"].fillna(pat["ndc"].map(name_map))
cols = ["rx_claim_id", "fill_date", "product_name", "status", "transaction_group_id", "reject_reason", "patient_pay"]
print(pat[cols].to_string(index=False))

rx_claim_id  fill_date product_name   status transaction_group_id                                          reject_reason  patient_pay
  RX0005501 2024-07-02     Roventra Rejected           RXG0004885 New-to-market review; exception documentation required         0.00
  RX0005502 2024-07-09     Roventra     Paid           RXG0004885                                                    NaN        49.43
  RX0005503 2024-08-09     Roventra     Paid           RXG0004886                                                    NaN        58.18
  RX0005504 2024-08-10     Roventra Reversed           RXG0004886                                                    NaN       -58.18
  RX0005505 2024-08-15     Roventra     Paid           RXG0004886                                                    NaN        58.18
  RX0005506 2024-09-09     Roventra     Paid           RXG0004887                                                    NaN        72.14
  RX0005507 2024-10-09     Roventra     Paid           RXG0004

**Listing 3.4: Derive completed fills from transaction groups**

In [7]:
pat_rx = rx.loc[rx.patient_id.eq("PAT02034") & rx.ndc.eq("90000-1001-11")].copy()
chains = (pat_rx.sort_values(["fill_date", "rx_claim_id"])
    .groupby("transaction_group_id", sort=False)
    .agg(first_date=("fill_date", "first"), statuses=("status", " -> ".join), net_patient_pay=("patient_pay", "sum"))
    .reset_index(drop=True))
chains["completed_fill"] = chains.statuses.str.endswith("Paid") & chains.net_patient_pay.gt(0)
print(chains.to_string(index=False))
print(f"completed_fills={int(chains.completed_fill.sum())}")

first_date                 statuses  net_patient_pay  completed_fill
2024-07-02         Rejected -> Paid            49.43            True
2024-08-09 Paid -> Reversed -> Paid            58.18            True
2024-09-09                     Paid            72.14            True
2024-10-09                     Paid            66.20            True
completed_fills=4


**Conclusion:** 7 transactions form 4 groups, and every group ends in a paid status with positive net patient payment. Count completed groups, not rows or paid statuses.

### 3.5.2 Formulary and access: what rules applied to PAT02034 in July?

In [8]:
patients = pd.read_csv(DATA / "reference" / "patients.csv")
fs = pd.read_csv(DATA / "formulary" / "formulary_status.csv")
fh = pd.read_csv(DATA / "formulary" / "formulary_history.csv")

payer = patients.loc[patients.patient_id.eq("PAT02034"), "payer_id"].iloc[0]
cols  = ["plan_id", "plan_name", "tier", "prior_authorization",
         "step_therapy", "quantity_limit", "specialty_pharmacy"]
status  = fs.loc[fs.plan_id.eq(payer) & fs.product_name.eq("Roventra"), cols]
changes = fh.loc[fh.plan_id.eq(payer) & fh.product_name.eq("Roventra")]
print(status.to_string(index=False))
print(f"Roventra change events for {payer}: {len(changes)}")

plan_id               plan_name      tier prior_authorization step_therapy quantity_limit specialty_pharmacy
 PAY002 South Commercial Plan 2 Specialty                 Yes          Yes            Yes                Yes
Roventra change events for PAY002: 0


**Conclusion:** PAY002 requires prior authorization, specialty-pharmacy routing, step therapy, and a quantity limit for Roventra. The rejected claim carries the event-specific new-to-market exception reason; the current status table carries the ongoing restrictions.

**Listing 3.5: Join a claim to the payer rule active on its date**

In [9]:
fh_dates = pd.read_csv(DATA / "formulary" / "formulary_history.csv",
                       parse_dates=["effective_date"])

pay001_hist = fh_dates.loc[fh_dates.plan_id.eq("PAY001") & fh_dates.product_name.eq("Roventra")].sort_values("effective_date")
print("PAY001 Roventra change history:")
print(pay001_hist[["effective_date","prior_tier","new_tier"]].to_string(index=False))
print()

initial_tier = pay001_hist.iloc[0]["prior_tier"]
initial_pa   = pay001_hist.iloc[0]["prior_prior_authorization"]
states = [{"effective_date": pd.Timestamp("2024-01-01"),
           "tier": initial_tier, "prior_authorization": initial_pa}]
for _, row in pay001_hist.iterrows():
    states.append({"effective_date": row["effective_date"],
                   "tier": row["new_tier"], "prior_authorization": row["new_prior_authorization"]})
state_df = pd.DataFrame(states)

def rule_on(event_date: str) -> dict:
    dt = pd.Timestamp(event_date)
    return state_df.loc[state_df.effective_date.le(dt)].iloc[-1].to_dict()

for date in ["2024-02-15", "2024-06-01"]:
    r = rule_on(date)
    print(f"{date}: tier={r['tier']}, PA={r['prior_authorization']}")

PAY001 Roventra change history:
effective_date prior_tier new_tier
    2024-04-01     Tier 3   Tier 1

2024-02-15: tier=Tier 3, PA=Yes
2024-06-01: tier=Tier 1, PA=Yes


**Key lesson:** reading only the current formulary status treats every historical claim as if the product were always at Tier 1. The as-of join is always required for event-date accuracy.

### 3.5.3 EHR: what does the chart add that claims cannot see?

In [10]:
ehr = pd.read_csv(DATA / "ehr" / "ehr_events.csv")
cols = ["event_date", "event_type", "event_name", "lab_value",
        "lab_abnormal_flag", "problem_list_codes", "medication_orders"]
ehr.loc[ehr.patient_id.eq("PAT02034"), cols].sort_values("event_date")

,event_date,event_type,event_name,lab_value,lab_abnormal_flag,problem_list_codes,medication_orders
15245,2024-03-07,Encounter,Office visit,NaN,NaN,E11.40|E11.9,Metformin 1000 mg oral BID
15246,2024-06-12,Encounter,Specialist visit,NaN,NaN,E11.40|E11.65,Roventra 10 mg oral daily
15247,2024-06-13,Lab,A1C,10.60,H,NaN,NaN
15248,2024-06-22,Lab,A1C,8.90,H,NaN,NaN
15249,2024-12-27,Encounter,Care coordination,NaN,NaN,E11.40|E11.65,Roventra 20 mg oral daily|Atorvastatin 40 mg o...
15250,2024-12-30,Lab,LDL,170.00,H,NaN,NaN
15251,2025-01-01,Lab,A1C,8.40,H,NaN,NaN


**Conclusion:** the clinical spine behind the claims. Diabetes appears in March, Roventra orders in June, first paid fill in July. An EHR medication order records treatment intent; pharmacy claims record observed dispensing.

### 3.5.4 CRM and digital: find the patient this call was about

In [11]:
crm = pd.read_csv(DATA / "crm_veeva" / "crm_interactions.csv")
print(list(crm.columns))
print()
cols = ["interaction_date", "channel", "message_topic", "hcp_id", "account_id"]
crm.loc[crm.hcp_id.eq("HCP0280") & crm.product_name.eq("Roventra"), cols].sort_values("interaction_date")

['crm_interaction_id', 'interaction_date', 'rep_id', 'hcp_id', 'account_id', 'territory', 'product_name', 'channel', 'message_topic', 'outcome', 'duration_min', 'sample_drops', 'consent_status', 'next_action']



,interaction_date,channel,message_topic,hcp_id,account_id
1179,2024-09-18,Email,Competitive differentiation,HCP0280,ACC089


**Conclusion:** there is no `patient_id` in this table. The CRM table is keyed to HCPs and accounts; patient-level attribution is outside its permitted grain. PAT02034's prescriber HCP0280 appears in CRM, but the interaction cannot be placed on PAT02034's timeline. This is both a privacy boundary and an analytical one.

### 3.5.5 Medicare Part D: highest-volume Roventra prescribers

In [12]:
partd = pd.read_csv(DATA / "cms_part_d" / "prescriber_summary.csv")
cols  = ["prscrbr_npi", "prscrbr_type", "prscrbr_city", "prscrbr_state_abrvtn",
         "tot_clms", "tot_benes", "tot_drug_cst"]
(
    partd.loc[partd.drug_name.eq("Roventra"), cols]
    .sort_values(["tot_clms", "tot_benes"], ascending=[False, False])
    .head(5)
)

,prscrbr_npi,prscrbr_type,prscrbr_city,prscrbr_state_abrvtn,tot_clms,tot_benes,tot_drug_cst
640,HCP0404,Pulmonology,NJ City,NJ,300,256,273837.90
535,HCP0337,Pulmonology,MI City,MI,298,144,94005.46
65,HCP0044,Endocrinology,TX City,TX,298,110,91179.53
961,HCP0619,Oncology/Hematology,NY City,NY,296,247,69255.49
233,HCP0150,Rheumatology,IL City,IL,294,243,70603.33


**Conclusion:** useful for ranking and benchmarking by specialty and geography. Absent: patient journeys, switching, payer, eligibility window, dates inside the year, or reason for the volume.

## 4. Two planted traps

See §3.7 of the chapter. The maturity trap was shown in section 2 above.

**Listing 3.6: Detect duplicate medical claims**

In [13]:
mc_dup = pd.read_csv(DATA / "claims_medical" / "medical_claims.csv")
keys = ["patient_id", "claim_date", "icd10_code", "procedure_code", "hcp_id", "paid_amount"]
detected = mc_dup.duplicated(subset=keys, keep="first").sum()
flagged   = mc_dup.is_duplicate.sum()
print(f"rows={len(mc_dup):,}  detected_by_keys={detected:,}  answer_key={flagged:,}")
print(f"naive count overstates utilization by {100*flagged/(len(mc_dup)-flagged):.2f}%")

rows=51,083  detected_by_keys=1,002  answer_key=1,002
naive count overstates utilization by 2.00%


**Listing 3.7: Restore missing product names from the NDC crosswalk**

In [14]:
rx_all = pd.read_csv(DATA / "claims_pharmacy" / "pharmacy_claims.csv", dtype={"ndc": str})
ndc_df = pd.read_csv(DATA / "reference" / "ndc_codes.csv", dtype={"NDC": str})
crosswalk = ndc_df.set_index("NDC")["CUI_L1_NAME"]

paid = rx_all.loc[rx_all.status.eq("Paid")].copy()
naive = paid.loc[paid.product_name.fillna("").str.strip().ne("")].groupby("product_name").size()

paid["product_name"] = paid["product_name"].fillna("").str.strip()
paid["product_name"] = paid["product_name"].mask(
    paid["product_name"].eq(""), paid["ndc"].map(crosswalk)
)
recovered = paid.groupby("product_name").size()

result = pd.DataFrame({
    "paid_fills_naive":     naive.reindex(recovered.index, fill_value=0).astype(int),
    "paid_fills_recovered": recovered.astype(int),
})
result["understated_pct"] = (100 * (1 - result.paid_fills_naive / result.paid_fills_recovered)).round(2)
result.sort_values("paid_fills_recovered", ascending=False)

,paid_fills_naive,paid_fills_recovered,understated_pct
product_name,,,
Roventra,15566,16361,4.86
Vexpro,10105,10608,4.74
Nexoral,9920,10448,5.05
Supportive Med,9816,10318,4.87


**Conclusion:** every brand was silently undercounted by ~5%. The naive table looks complete. The rule: recover what the reference can, keep the remainder visible as unmapped.

## 5. Quality gate

In [15]:
import sys
sys.path.insert(0, str(SCRIPTS))
from data_quality import run_data_quality

_, summary = run_data_quality(DATA)
summary

,metric,value,unit
0,Duplicate medical claims,1.96,%
1,Median claim receipt lag,12.00,days
2,Missing product mappings,4.92,%
3,Taxonomy conflicts or gaps,3.15,%
4,Eligible observation window,51.45,%


**Conclusion:** duplicates, mapping gaps, and taxonomy conflicts are defect checks. The observation-window result is an eligibility count whose value changes with the study window.

## 6. Laboratory results to patient groups

**Listing 3.8: Build A1C patient groups without dropping no-result patients**

In [16]:
ehr_all = pd.read_csv(DATA / "ehr" / "ehr_events.csv", parse_dates=["event_date"])
patients_all = pd.read_csv(DATA / "reference" / "patients.csv")
a1c = ehr_all.loc[ehr_all.event_name.eq("A1C")].copy()
a1c["lab_value"] = pd.to_numeric(a1c["lab_value"], errors="coerce")
latest = (a1c.dropna(subset=["patient_id", "event_date", "lab_value"])
    .sort_values(["patient_id", "event_date", "lab_value"])
    .groupby("patient_id", as_index=False).tail(1)[["patient_id", "lab_value"]])
latest["a1c_class"] = pd.cut(latest.lab_value, [-float("inf"), 5.7, 6.5, float("inf")], labels=["normal", "prediabetes", "diabetes"], right=False).astype("string")
cohort = patients_all[["patient_id"]].merge(latest, on="patient_id", how="left")
cohort["a1c_class"] = cohort["a1c_class"].fillna("no_result")
summary = cohort.groupby("a1c_class", sort=False).size().rename("patient_count").reset_index()
summary["pct"] = (100 * summary.patient_count / len(patients_all)).round(2)
print(summary.sort_values("patient_count", ascending=False).to_string(index=False))

  a1c_class  patient_count   pct
   diabetes           8699 43.50
  no_result           5576 27.88
     normal           3108 15.54
prediabetes           2617 13.08


**Three conclusions:**

1. **The denominator lesson.** 43.5% of *all* patients are in the diabetes band, but 27.9% have no result. Restrict to tested patients and "diabetes" jumps to 60.3% -- a different answer to a different question.

2. **Missingness has structure.** PSA is 92.8% missing because the generator orders it only for male oncology patients. Who gets tested is informative.

3. **Prevalence is a scenario assumption.** This synthetic population is enriched for the launch condition. Chapter 4 calibrates against real prevalence anchors.

**Chapter summary:** see the 8-question checklist at the end of §3.10 of the chapter manuscript.